In [1]:
import numpy as np
import pandas as pd

from pathlib import Path

In [3]:
try:
    df = pd.read_csv(
        "../data/raw/sp500.csv",
        parse_dates=["Date"],
        index_col="Date",
    )
except ValueError:
    # fallback: parse the first column as dates and use it as the index
    df = pd.read_csv(
        "../data/raw/sp500.csv",
        parse_dates=[0],
        index_col=0,
    )
    df.index.name = "Date"

df.head()

/tmp/ipykernel_7390/2919963639.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(


,Close
Date,
Ticker,^GSPC
Date,NaN
2000-01-03,1455.219970703125
2000-01-04,1399.4200439453125
2000-01-05,1402.1099853515625


In [4]:
# sort by date
df = df.sort_index()

# remove missing values
df = df.dropna()

df.head()

,Close
Date,
2000-01-03,1455.219970703125
2000-01-04,1399.4200439453125
2000-01-05,1402.1099853515625
2000-01-06,1403.449951171875
2000-01-07,1441.469970703125


In [6]:
df["Close"] = pd.to_numeric(df["Close"], errors="coerce")

df["LogReturn"] = np.log(
    df["Close"] / df["Close"].shift(1)
)

df = df.dropna()

df.head()

,Close,LogReturn
Date,,
2000-01-04,1399.420044,-0.039099
2000-01-05,1402.109985,0.001920
2000-01-06,1403.449951,0.000955
2000-01-07,1441.469971,0.026730
2000-01-10,1457.599976,0.011128


In [7]:
split_idx = int(len(df) * 0.8)

train = df.iloc[:split_idx].copy()

test = df.iloc[split_idx:].copy()

print(len(train))
print(len(test))

5230
1308


In [8]:
train_mean = train["LogReturn"].mean()

train_std = train["LogReturn"].std()

train["Normalized"] = (
    train["LogReturn"] - train_mean
) / train_std

test["Normalized"] = (
    test["LogReturn"] - train_mean
) / train_std

In [9]:
WINDOW_SIZE = 30

In [10]:
def create_windows(series, window_size):
    windows = []

    for i in range(len(series) - window_size):
        windows.append(
            series[i:i+window_size]
        )

    return np.array(windows)

In [11]:
train_windows = create_windows(
    train["Normalized"].values,
    WINDOW_SIZE,
)

test_windows = create_windows(
    test["Normalized"].values,
    WINDOW_SIZE,
)

print(train_windows.shape)
print(test_windows.shape)

(5200, 30)
(1278, 30)


In [12]:
Path("../data/processed").mkdir(
    parents=True,
    exist_ok=True,
)

In [13]:
train.to_csv("../data/processed/train.csv")

test.to_csv("../data/processed/test.csv")

In [14]:
np.save(
    "../data/processed/train_windows.npy",
    train_windows,
)

np.save(
    "../data/processed/test_windows.npy",
    test_windows,
)

In [15]:
print(train.head())

print()

print(test.head())

print()

print(train_windows.shape)

print(test_windows.shape)

                  Close  LogReturn  Normalized
Date                                          
2000-01-04  1399.420044  -0.039099   -3.122499
2000-01-05  1402.109985   0.001920    0.139437
2000-01-06  1403.449951   0.000955    0.062689
2000-01-07  1441.469971   0.026730    2.112336
2000-01-10  1457.599976   0.011128    0.871631

                  Close  LogReturn  Normalized
Date                                          
2020-10-16  3483.810059   0.000135   -0.002543
2020-10-19  3426.919922  -0.016465   -1.322567
2020-10-20  3443.120117   0.004716    0.361768
2020-10-21  3435.560059  -0.002198   -0.188069
2020-10-22  3453.489990   0.005205    0.400667

(5200, 30)
(1278, 30)
